# Agentic Payments Walkthrough

This notebook condenses the seven-script `agentic-payments/` walkthrough into a single narrative. It demonstrates the three primitives an agentic cross-border payment system needs in order to be reproducible 18 months after the fact:

1. **Bitemporal evidence** — every fact carries `valid_time` (when it was true) and `transaction_time` (when the system learned about it). Corrections append, never mutate.
2. **Versioned routing policy** — the agent's "if this use case, then this stablecoin" rules are themselves stored bitemporally, so an examiner can ask which policy was live on any given day.
3. **The examiner bundle** — a content-addressed JSON artifact joining the decision, the policy-as-of-decision, and the evidence records. `verify()` detects tamper.

> Everything below runs fully offline against in-memory stores. No external services, no API keys. Install the SDK first: `pip install "briefcase-ai[bitemporal,compliance,routing,external]>=3.0.0"`.

## Setup

Normalize the working directory so `_bootstrap` and `data.seed` import cleanly whether the notebook is launched from the repo root or from `agentic-payments/`.

In [ ]:
import os, sys

# Launchable from the repo root OR from agentic-payments/.
if not os.path.exists("_bootstrap.py") and os.path.exists("agentic-payments/_bootstrap.py"):
    os.chdir("agentic-payments")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import _bootstrap  # noqa: F401 — stubs briefcase._native if not compiled

import json
import briefcase
from briefcase.bitemporal import (
    AsOfView,
    BitemporalRecord,
    InMemoryBitemporalStore,
    append_correction,
)
from briefcase.compliance import BundleIntegrityError, ExaminerBundle
from briefcase.routing import AgentRouter, PolicyRegistry

from data.seed import (
    at,
    bloomberg_original_record,
    describe_record,
    ofac_clean_record,
    print_table,
    stablecoin_policy_v1,
    stablecoin_policy_v2,
)

print(f"Briefcase AI SDK: {getattr(briefcase, '__version__', 'unknown')}")
print(f"Working directory: {os.getcwd()}")

## 1. Bitemporal basics

`BitemporalRecord` is the evidence atom. The first cell below constructs one directly via `BitemporalRecord.new()` so the full shape is visible. Every record carries:

| Field | Meaning |
|---|---|
| `key` | The logical entity the record is about (`ofac:cp-42`, `USDC/USD`, …) |
| `valid_time` | When the fact was true in the world (e.g. when the FX print was observed) |
| `transaction_time` | When the system learned about it (e.g. when it was ingested) |
| `value` | The payload — any JSON-serializable dict |
| `source`, `source_trust_level` | Attribution — who said this, and how much to trust them |
| `metadata` | Free-form; often includes feed version, upstream commit SHA, etc. |
| `parent_record_id` | `None` for originals; set on corrections (§2) |

`InMemoryBitemporalStore` exposes `append()`, `history()`, `latest()`, `as_of()`, and `keys()`. There is deliberately no `update()` — the store is append-only by construction, which is what makes §3's replay guarantee hold.

In [ ]:
# Construct a BitemporalRecord directly to expose its anatomy.
demo_record = BitemporalRecord.new(
    key="ofac:cp-42",
    valid_time=at(16),
    value={"sanctioned": False, "jurisdiction": "US"},
    source="ofac",
    source_trust_level="primary",
    transaction_time=at(16),
    metadata={"sdn_list_version": "2026-04-17"},
)

print("BitemporalRecord.new(...) produces:")
print(f"  record_id:         {demo_record.record_id}")
print(f"  key:               {demo_record.key}")
print(f"  valid_time:        {demo_record.valid_time.isoformat()}")
print(f"  transaction_time:  {demo_record.transaction_time.isoformat()}")
print(f"  value:             {demo_record.value}")
print(f"  source:            {demo_record.source} (trust={demo_record.source_trust_level})")
print(f"  metadata:          {demo_record.metadata}")
print(f"  parent_record_id:  {demo_record.parent_record_id}  (None → original, not a correction)")

print("\n" + "-" * 60 + "\n")

# The rest of the walkthrough uses the factory helpers in data/seed.py
# (ofac_clean_record, bloomberg_original_record, …) to keep cells short.
store = InMemoryBitemporalStore()
store.append(ofac_clean_record())
store.append(bloomberg_original_record())

print("Store keys:")
for k in store.keys():
    print(f"  - {k}")

print("\nFull history per key:")
for k in store.keys():
    print(f"\nKey: {k}")
    print_table([describe_record(r) for r in store.history(k)])

print(f"\nTotal rows: {len(store)}")

## 2. The correction pattern

Bloomberg publishes a price. Thirty days later Bloomberg discovers an error and issues a correction. The architecturally correct response is **not** to overwrite the original — it is to append a new record with the same `valid_time` and a fresh `transaction_time`. The correction links to the original via `parent_record_id`.

In [ ]:
store = InMemoryBitemporalStore()
original = bloomberg_original_record()
store.append(original)

print("Initial state:")
print_table([describe_record(r) for r in store.history("USDC/USD")])

correction = append_correction(
    store,
    original,
    corrected_value={"px": 1.0002, "size": 1_000_000},
    transaction_time=at(46),  # 30 days after the original
)

print("\nAfter the correction is appended:")
print_table([describe_record(r) for r in store.history("USDC/USD")])

print(f"\n  Original record_id:       {original.record_id}")
print(f"  Correction record_id:     {correction.record_id}")
print(f"  Correction.parent_record: {correction.parent_record_id}")
print(f"  latest().value:           {store.latest('USDC/USD').value}")

## 3. AsOfView — replay without rebuilding

The examiner asks, months later: *"what did your system believe on day 30?"* `AsOfView` wraps any bitemporal store and clamps reads to a historical `transaction_time`. Application code does not change between live operation and replay — only the clamp changes. Writes are refused on a view of the past.

In [ ]:
# Same store as §2: original on day 16, correction on day 46.
store = InMemoryBitemporalStore()
original = bloomberg_original_record()
store.append(original)
append_correction(
    store, original,
    corrected_value={"px": 1.0002, "size": 1_000_000},
    transaction_time=at(46),
)

print(f"Live view (no clamp):                 {store.latest('USDC/USD').value}")

with AsOfView(store, transaction_time=at(30)) as view:
    print(f"As-of day 30 (before correction):     {view.latest('USDC/USD').value}")

with AsOfView(store, transaction_time=at(50)) as view:
    print(f"As-of day 50 (after correction):      {view.latest('USDC/USD').value}")

# Writes on a view of the past are refused.
view = AsOfView(store, transaction_time=at(30))
try:
    view.append(original)
except Exception as e:
    print(f"\nWrite on AsOfView refused: {type(e).__name__}: {e}")

## 4. Versioned routing policy

The routing rules (*"LATAM payouts use USDT for liquidity"*) are themselves stored bitemporally. Publishing v2 on day 60 does not erase v1 — the registry keeps both. Reading "the policy as-of day 50" returns v1; "as-of day 90" returns v2. Identical call site, different clamp, different answer.

In [ ]:
def describe_policy(policy) -> None:
    if policy is None:
        print("  (no policy visible as-of that date)")
        return
    print(f"  version:        {policy.version}")
    print(f"  description:    {policy.description}")
    print(f"  rules ({len(policy.rules)}):")
    for rule in policy.rules:
        print(f"    - {rule.rule_id}: {rule.condition} -> {rule.choice}")
    print(f"  default_choice: {policy.default_choice}")

registry = PolicyRegistry()
v1 = stablecoin_policy_v1()
registry.publish(v1, valid_from=at(0), transaction_time=at(0))
v2 = stablecoin_policy_v2()
registry.publish(v2, valid_from=at(60), transaction_time=at(60))

print("Registry history for 'stablecoin_router':")
for p in registry.history("stablecoin_router"):
    print(f"  {p.version}: {p.description}")

print("\n--- As-of day 50 (before v2 published) ---")
describe_policy(registry.get("stablecoin_router", as_of_transaction_time=at(50)))

print("\n--- As-of day 90 (after v2 published) ---")
describe_policy(registry.get("stablecoin_router", as_of_transaction_time=at(90)))

## 5. Agent routing with evidence attribution

`AgentRouter` joins a policy registry with an evidence store: given a context, it selects the policy-as-of the clamp, matches a rule, and emits a decision record carrying the matched rule, the policy version, and the evidence record IDs that justified the inputs. Same context, different clamp → different (reproducible) decision.

In [ ]:
# Evidence store and policy registry.
evidence = InMemoryBitemporalStore()
ofac = ofac_clean_record()
fx = bloomberg_original_record()
evidence.append(ofac)
evidence.append(fx)

registry = PolicyRegistry()
registry.publish(stablecoin_policy_v1(), valid_from=at(0), transaction_time=at(0))
registry.publish(stablecoin_policy_v2(), valid_from=at(60), transaction_time=at(60))

router = AgentRouter(
    registry,
    use_case="cross_border_payout",
    policy_id="stablecoin_router",
)

context = {
    "jurisdiction": "LATAM",
    "sanctioned": ofac.value["sanctioned"],
    "notional_usd": 250_000,
}
evidence_refs = [ofac.record_id, fx.record_id]

print("=== Route today (v2 live) ===")
decision_today = router.route(context, evidence_refs=evidence_refs)
print(f"  selected:        {decision_today.selected}")
print(f"  policy_version:  {decision_today.policy_version}")
print(f"  matched_rule_id: {decision_today.matched_rule_id}")
print(f"  rationale:       {decision_today.rationale}")

print("\n=== Same context, clamped to day 30 (v1 was live) ===")
decision_then = router.route(
    context, evidence_refs=evidence_refs, as_of_transaction_time=at(30)
)
print(f"  selected:        {decision_then.selected}")
print(f"  policy_version:  {decision_then.policy_version}")
print(f"  matched_rule_id: {decision_then.matched_rule_id}")
print(f"  rationale:       {decision_then.rationale}")

## 6. The look-ahead trap

A naive backtest reads the current store and asks "what did the system think on day 17?" — but the store has already absorbed corrections that did not exist on day 17. The backtest sees the future; the Sharpe ratio lifts; the trader loses money in production.

`AsOfView` closes the trap. **Same function body, same store, same query — only the wrapper differs.** That property ("backtest is production with a clamp") is what makes the guarantee hold in the general case.

In [ ]:
def naive_backtest(store, day: int) -> dict:
    # BUG: reads the live store, which contains corrections appended after `day`.
    return store.latest("USDC/USD").value

def asof_backtest(store, day: int) -> dict:
    with AsOfView(store, transaction_time=at(day)) as view:
        return view.latest("USDC/USD").value

store = InMemoryBitemporalStore()
original = bloomberg_original_record()
store.append(original)
append_correction(
    store, original,
    corrected_value={"px": 1.0002, "size": 1_000_000},
    transaction_time=at(46),
)

print("Backtest question: 'what was the px on day N?'\n")
for day in (20, 30, 45, 50):
    naive = naive_backtest(store, day)
    correct = asof_backtest(store, day)
    tag = "   LEAK" if naive != correct else "  clean"
    print(f"  day {day:>3}: naive={naive['px']:.4f}  as-of={correct['px']:.4f}  [{tag}]")

## 7. The examiner bundle

The payoff. Months after a decision ships, an examiner asks: *"reproduce this routing call, and prove the artifact you're handing me is the one you actually had at the time."*

`ExaminerBundle.build()` joins the decision, the policy-as-of-the-decision, and the evidence records into a single JSON payload with a SHA-256 content hash. `verify()` detects any tamper — and survives a JSON round-trip.

In [ ]:
# Evidence + policy registry (same setup as §5).
evidence = InMemoryBitemporalStore()
ofac = ofac_clean_record()
fx = bloomberg_original_record()
evidence.append(ofac)
evidence.append(fx)

registry = PolicyRegistry()
registry.publish(stablecoin_policy_v1(), valid_from=at(0), transaction_time=at(0))
registry.publish(stablecoin_policy_v2(), valid_from=at(60), transaction_time=at(60))

# Produce a decision as if it happened on day 30 (under v1).
router = AgentRouter(registry, use_case="cross_border_payout", policy_id="stablecoin_router")
context = {
    "jurisdiction": "LATAM",
    "sanctioned": ofac.value["sanctioned"],
    "notional_usd": 250_000,
}
decision = router.route(
    context,
    evidence_refs=[ofac.record_id, fx.record_id],
    as_of_transaction_time=at(30),
)
decision.decided_at = at(30)

print(f"Decision selected:       {decision.selected}")
print(f"Decision policy_version: {decision.policy_version}")

# Build and fingerprint the bundle.
bundle = ExaminerBundle.build(
    decision,
    evidence_store=evidence,
    policy_registry=registry,
    metadata={"use_case": decision.use_case, "notional_usd": context["notional_usd"]},
)

print("\n=== Bundle summary ===")
print(f"  schema_version:         {bundle.schema_version}")
print(f"  as_of_transaction_time: {bundle.as_of_transaction_time}")
print(f"  policy in bundle:       {bundle.policy['version']}")
print(f"  evidence rows:          {len(bundle.evidence)}")
print(f"  content_hash:           {bundle.content_hash}")

bundle.verify()
print("\nverify() on the untouched bundle: OK")

payload = bundle.to_json(indent=2)
rehydrated = ExaminerBundle.from_json(payload)
rehydrated.verify()
print("verify() after JSON round-trip:   OK")

# Tamper check — swap USDT for USDC in the decision.
tampered_dict = json.loads(payload)
tampered_dict["decision"]["selected"] = "USDC"
tampered = ExaminerBundle.from_dict(tampered_dict)
try:
    tampered.verify()
except BundleIntegrityError as e:
    print(f"verify() on tampered bundle:      REJECTED ({type(e).__name__})")
    print(f"   └─ {str(e).splitlines()[0]}")

## Key takeaways

- **`BitemporalRecord` + `InMemoryBitemporalStore`** — evidence is append-only, carries both world-time and system-time, and cannot be silently mutated.
- **`append_correction()`** — corrections are first-class records with a `parent_record_id` back-pointer; the original belief is preserved.
- **`AsOfView`** — the single primitive that makes historical replay trivial. Same code path, writes refused, deterministic results.
- **`PolicyRegistry`** — routing rules stored bitemporally, so you can always ask *"which policy was live on day X?"*.
- **`AgentRouter`** — joins evidence + policy and emits a decision record whose every input is a bitemporal reference.
- **`ExaminerBundle`** — a content-addressed, self-contained, tamper-evident artifact. Survives JSON round-trip; `verify()` fails hard on any mutation.

### The invariant to remember

Naive: `store.latest(key)` — reads the world as it is *now*.
Correct: `AsOfView(store, t).latest(key)` — reads the world as it was known at time *t*.

For any decision whose inputs have since been corrected, these two return different values. The difference is the look-ahead bias — and it is exactly the thing an auditor will ask you to prove is not there.

### Where to go next

**To see each primitive in isolation** (domain-neutral, single-file demos):

- [`patterns/02_bitemporal_evidence.py`](../patterns/02_bitemporal_evidence.py) — `BitemporalRecord` + `InMemoryBitemporalStore`
- [`patterns/03_correction_append.py`](../patterns/03_correction_append.py) — `append_correction`
- [`patterns/04_temporal_replay.py`](../patterns/04_temporal_replay.py) — `AsOfView`
- [`patterns/05_versioned_policy.py`](../patterns/05_versioned_policy.py) — `PolicyRegistry`
- [`patterns/06_examiner_bundle.py`](../patterns/06_examiner_bundle.py) — `ExaminerBundle`
- [`patterns/patterns_walkthrough.ipynb`](../patterns/patterns_walkthrough.ipynb) — the full 11-pattern unified tour

**To see these primitives integrated into other domains** (replay-layer capstones in each):

- [`regulatory-workflows/02_ofac_sanctions/`](../regulatory-workflows/02_ofac_sanctions/) — sanctions screening, SDN delisting correction
- [`regulatory-workflows/01_credit_underwriting/`](../regulatory-workflows/01_credit_underwriting/) — credit decisioning, FCRA dispute correction
- [`regulatory-workflows/05_mortgage_fair_lending/`](../regulatory-workflows/05_mortgage_fair_lending/) — fair lending, appraisal revision
- [`regulatory-workflows/07_aml_transaction_monitoring/`](../regulatory-workflows/07_aml_transaction_monitoring/) — AML, beneficial-owner restatement
- [`regulatory-workflows/14_algo_trading_surveillance/`](../regulatory-workflows/14_algo_trading_surveillance/) — surveillance, exchange print bust

Each of those examples keeps its existing decision-capture story and appends a **"BITEMPORAL REPLAY DEMONSTRATION"** section showing how the same primitives answer the question *"prove what was known on decision day, not what is known now."* The correction scenario is authentic to each domain — the same SDK primitive handles five different shapes of upstream restatement.